# CIFAR-10 Image Classification using a Feedforward Neural Network (FNN)

This notebook follows the supplied code cell by cell. It loads CIFAR-10, normalizes the images, trains an FNN, displays a confusion matrix, evaluates the model, and plots training/validation accuracy.

**Note:** The first run downloads CIFAR-10 and training can take several minutes. The notebook contains executable cells; outputs will appear after you run each cell in Jupyter or Colab.

## Cell 1 — Import libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import seaborn as sns
from sklearn.metrics import confusion_matrix

print("TensorFlow version:", tf.__version__)

## Cell 2 — Load the CIFAR-10 dataset

In [ ]:
# data = pd.read_csv("cifar-10.csv")

# print(data.head())
# print("Dataset Shape:", data.shape)
(X_train, y_train), (X_test, y_test) = tf.keras.datasets.cifar10.load_data()

print("Training images shape:", X_train.shape)
print("Training labels shape:", y_train.shape)
print("Testing images shape:", X_test.shape)
print("Testing labels shape:", y_test.shape)

## Cell 3 — Define class names

In [ ]:
classes = ['Airplane', 'Automobile', 'Bird', 'Cat', 'Deer',
           'Dog', 'Frog', 'Horse', 'Ship', 'Truck']

print("Number of classes:", len(classes))
print(classes)

## Cell 4 — Convert labels into 1D arrays

In [ ]:
y_train = y_train.flatten()
y_test = y_test.flatten()

print("Flattened training labels:", y_train.shape)
print("Flattened testing labels:", y_test.shape)

## Cell 5 — Normalize and reshape images

In [ ]:
# Each CIFAR-10 image is 32 x 32 pixels with 3 color channels.
# Flattening gives 32 * 32 * 3 = 3072 features per image.
# Dividing by 255 scales pixel values from [0, 255] to [0, 1].

X_train = X_train.reshape(-1, 3072).astype("float32") / 255.0
X_test = X_test.reshape(-1, 3072).astype("float32") / 255.0

print("Processed training shape:", X_train.shape)
print("Processed testing shape:", X_test.shape)
print("Pixel value range:", X_train.min(), "to", X_train.max())

## Cell 6 — Build the feedforward neural network (FNN)

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(3072,)),
    tf.keras.layers.Dense(512, activation="relu"),
    tf.keras.layers.Dense(256, activation="relu"),
    tf.keras.layers.Dense(10, activation="softmax")
])

model.summary()

## Cell 7 — Compile the model

In [ ]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("Model compiled successfully.")

## Cell 8 — Train the model

In [ ]:
history = model.fit(
    X_train, y_train,
    epochs=10,
    batch_size=128,
    validation_split=0.1
)

## Cell 9 — Predict labels for test images

In [ ]:
test_probabilities = model.predict(X_test, batch_size=256, verbose=1)
y_pred = np.argmax(test_probabilities, axis=1)

print("Predicted labels shape:", y_pred.shape)
print("First 10 actual labels:   ", y_test[:10])
print("First 10 predicted labels:", y_pred[:10])

## Cell 10 — Display the confusion matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=classes,
    yticklabels=classes
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("CIFAR-10 Confusion Matrix")
plt.tight_layout()
plt.show()

## Cell 11 — Evaluate the model on the test set

In [ ]:
loss, accuracy = model.evaluate(X_test, y_test, verbose=1)

print(f"Test Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f} ({accuracy * 100:.2f}%)")

## Cell 12 — Plot training and validation accuracy

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history.history["accuracy"], marker="o")
plt.plot(history.history["val_accuracy"], marker="o")
plt.title("Model Accuracy")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.legend(["Train", "Validation"])
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Optional — Display a few predictions

This extra cell helps you visually check how the classifier performs on individual CIFAR-10 images. It reshapes the flattened test images back to 32×32×3 for display.

In [ ]:
# Recover the original RGB image shape for visualization.
# X_test values are normalized, so they can be displayed directly.
X_test_images = X_test.reshape(-1, 32, 32, 3)

plt.figure(figsize=(12, 6))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(X_test_images[i])
    actual_name = classes[int(y_test[i])]
    predicted_name = classes[int(y_pred[i])]
    color = "green" if y_test[i] == y_pred[i] else "red"
    plt.title(f"Actual: {actual_name}\nPred: {predicted_name}", color=color, fontsize=9)
    plt.axis("off")
plt.tight_layout()
plt.show()